### Wave B — Funnel & status

Status = current `leadstagehistories` (`iscurrent`) → `statusid` → `leadstatuses.name`.

| Sheet | Yellow box |
|-------|------------|
| `Channel_x_Status` | #16 |
| `Status_Distribution` | #27 |
| `Status_By_Month` | #28 |
| `Booking_vs_Lead` | #15 |
| `ContactType_to_Booking` | #14 |
| `Bookings_By_Channel` | #29 |
| `Bookings_By_Source` | #32–34 |



In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path("/Users/lekhanigam/data/leads_sv_closure_reports")
sys.path.insert(0, str(ROOT))
from common import START_TS, END_TS, CLOSURE_STAGE, q, write_excel

leads = q(f"""
WITH ct AS (
  SELECT CAST(cid AS varchar) cid, MAX(CAST(source AS varchar)) src
  FROM campaigntrackers WHERE cid IS NOT NULL GROUP BY 1
),
cur AS (
  SELECT CAST(h.lead AS varchar) lead_oid,
         CAST(h.statusid AS varchar) statusid,
         CAST(h.status AS varchar) status_code,
         ROW_NUMBER() OVER (PARTITION BY CAST(h.lead AS varchar) ORDER BY h.createdat DESC) rn
  FROM leadstagehistories h
  WHERE CAST(h.iscurrent AS varchar)='true'
    AND h.createdat >= {START_TS}
)
SELECT
  CAST(l._id AS varchar) lead_oid,
  date_format(l.createdat, '%Y-%m') month_key,
  date_format(l.createdat, '%M') month_name,
  TRIM(CAST(ls.name AS varchar)) source,
  COALESCE(NULLIF(TRIM(CAST(us.name AS varchar)), ''), TRIM(CAST(ls.name AS varchar))) channel,
  COALESCE(NULLIF(TRIM(CAST(st.name AS varchar)), ''), NULLIF(TRIM(c.status_code), '')) status_name,
  NULLIF(TRIM(CAST(l.creationtype AS varchar)), '') contact_type
FROM leads l
LEFT JOIN leadsources ls ON CAST(ls._id AS varchar)=CAST(l.bornsource AS varchar)
LEFT JOIN ct ON ct.cid=TRIM(CAST(l.utm_campaign AS varchar))
LEFT JOIN utmsources us ON CAST(us._id AS varchar)=ct.src
LEFT JOIN cur c ON c.lead_oid=CAST(l._id AS varchar) AND c.rn=1
LEFT JOIN leadstatuses st ON CAST(st._id AS varchar)=c.statusid
WHERE l.createdat >= {START_TS} AND l.createdat < {END_TS}
""")
closed = q(f"""
SELECT CAST(lead AS varchar) lead_oid FROM leadstagehistories
WHERE CAST(stage AS varchar)='{CLOSURE_STAGE}' AND createdat >= {START_TS} AND createdat < {END_TS}
GROUP BY 1
""")
leads["is_closed"] = leads["lead_oid"].isin(set(closed["lead_oid"])).astype(int)

def with_conv(df):
    out = df.copy()
    out["Conversion rate %"] = np.where(out["Lead count"]>0, (100*out["Sales closure count"]/out["Lead count"]).round(2), 0)
    return out

sheets = {
    "Channel_x_Status": leads.groupby(["channel","status_name"], as_index=False).agg(**{"Lead count":("lead_oid","nunique")}).sort_values("Lead count", ascending=False),
    "Status_Distribution": leads.groupby(["status_name"], as_index=False).agg(**{"Lead count":("lead_oid","nunique")}).sort_values("Lead count", ascending=False),
    "Status_By_Month": leads.groupby(["month_name","month_key","status_name"], as_index=False).agg(**{"Lead count":("lead_oid","nunique")}).sort_values(["month_key","Lead count"], ascending=[True,False]).drop(columns=["month_key"]),
    "Booking_vs_Lead": leads.groupby(["month_name","month_key"], as_index=False).agg(**{"Lead count":("lead_oid","nunique"),"Sales closure count":("is_closed","sum")}).sort_values("month_key").drop(columns=["month_key"]),
    "ContactType_to_Booking": with_conv(leads.groupby(["contact_type"], as_index=False).agg(**{"Lead count":("lead_oid","nunique"),"Sales closure count":("is_closed","sum")})).sort_values("Lead count", ascending=False),
    "Bookings_By_Channel": leads.groupby(["channel"], as_index=False).agg(**{"Lead count":("lead_oid","nunique"),"Sales closure count":("is_closed","sum")}).sort_values("Sales closure count", ascending=False),
    "Bookings_By_Source": with_conv(leads.groupby(["source"], as_index=False).agg(**{"Lead count":("lead_oid","nunique"),"Sales closure count":("is_closed","sum")})).sort_values("Sales closure count", ascending=False),
}
write_excel(ROOT / "Lead_Funnel_Status_Analysis.xlsx", sheets)
print("leads", len(leads), "closures", int(leads.is_closed.sum()))

